# ChipWhisperer 입문 강의자료 — 1강

## Side-channel analysis / SCA(부채널 분석) 플랫폼 실습

---

### 🎯 강의 목표

이 노트북은 **ChipWhisperer 플랫폼을 처음 접하는 사용자**를 대상으로 합니다.
부채널 분석 실험의 출발점인 **"전력 Trace(트레이스)를 안정적으로 수집하기"**까지를 단계별로 학습합니다.

| 단계 | 내용 | 핵심 산출물 |
|:----:|:----|:----|
| **1단계** | ChipWhisperer ↔ 타겟 보드 데이터 송수신 | SimpleSerial 통신 검증(Golden model(골든 모델) 일치) |
| **2단계** | Trace 수집 파라미터 이해 | `trig_count` 측정 → `samples` 결정 |
| **3단계** | 단일 Trace 수집 및 시각화 | Trace 1개 확인 |
| **4단계** | 다량 Trace 수집 + HDF5 파일 저장 | `*.h5` 파일 (수백~수만 개 Trace) |
| **5단계** | HDF5 파일 불러오기 및 Dataset 검증 | 저장된 데이터 재확인 + 시각화 |

> 본 강의는 점진적으로 보강되는 자료이므로, 셀 단위로 차근차근 실행하며 각 단계의 의미를 이해하는 것이 중요합니다.

---

## 🔍 시작하기 전에

### 부채널 분석(SCA)이란?

암호 알고리즘이 수학적으로는 안전하더라도, 실제 **하드웨어가 연산을 수행할 때 발생하는 부수적 정보**(전력 소비, 전자기 방사, 처리 시간 등)에는 비밀 키와 관련된 정보가 누설될 수 있습니다.
이러한 누설을 측정·분석해서 키를 복원하는 기법을 **부채널 분석(Side-Channel Analysis, SCA)** 이라 부릅니다.

전력 분석이 가능한 이유는, CMOS 회로의 스위칭 활동이 처리 중인 중간값(intermediate value)의 해밍 가중치/거리에 따라 달라지기 때문입니다.
즉, **"같은 키 + 다른 평문"** 으로 여러 번 측정하면, 각 시점의 전력 소비와 중간값 사이에 통계적 상관관계가 나타납니다.

### 왜 ChipWhisperer를 쓰는가?

부채널 측정은 본래 고가의 오실로스코프 + 정밀 트리거 환경이 필요하지만, **ChipWhisperer**는 다음을 한 보드에 통합한 오픈소스 플랫폼입니다:

- 타겟 MCU 클럭에 **동기화된 ADC** (jitter 최소화)
- 측정 트리거 + 타겟 통신 + 전원 분석을 **USB 한 줄**로 처리
- Python API로 손쉬운 자동화

### 실험 환경

```
호스트 PC (Python / Jupyter)
    │
    │  USB
    ▼
ChipWhisperer-Husky (scope + 제어보드)
    │
    │  20-pin 커넥터
    ▼
CW308 UFO 보드 — STM32F303 마이크로컨트롤러 (타겟)
```

- **scope** : 전력 Trace를 측정하는 오실로스코프 객체
- **target** : 타겟 MCU와 SimpleSerial로 통신하는 객체
- 타겟 펌웨어는 호스트가 보낸 명령어를 해석해 연산을 수행하고, 결과를 다시 돌려보냅니다.

### 📖 핵심 용어집

| 용어 | 의미 |
|:----|:----|
| **Trace(트레이스)** | 한 Execution(실행)에서 시간에 따라 측정한 타겟의 전력 벡터 |
| **Trigger** | 측정 시작/종료 시점을 알려주는 GPIO 신호 |
| **trig_count** | 트리거 ON → OFF 사이에 ADC가 수집한 샘플 개수 |
| **samples** | Trace 하나에 저장할 Sample(샘플)의 수 |
| **presamples** | 트리거 직전에 추가로 수집할 샘플 수 (배경 노이즈 관찰용) |
| **decimate** | 다운샘플링 비율 (1=원본, 2=절반, …) |
| **TV (Test Vector)** | 측정에 사용할 입력값 세트 (`fixed` 또는 `random`) |
| **Golden model(골든 모델)** | 호스트 PC에서 동일 연산을 직접 계산한 기준 출력값 |
| **HDF5** | 대용량 Dataset을 저장하는 계층적 바이너리 파일 포맷 |
| **CPA / DPA** | 다음 강의에서 다룰 통계적 부채널 공격 기법 |

---

## 📦 사전 준비: 플랫폼 설정 및 시각화 환경

### PLATFORM 변수 설정

사용하는 ChipWhisperer 하드웨어 종류를 지정합니다.
본 실습은 **HUSKY + CW308_STM32F3** 환경을 기준으로 합니다.

In [ ]:
# 사용 중인 ChipWhisperer 플랫폼을 지정합니다
# CW308_STM32F3 : CW308 UFO 보드 + STM32F303 타겟 (본 실습 기본 설정)
# CWHUSKY       : ChipWhisperer-Husky with CW313 + SAM4S 사용 시

PLATFORM = 'CW308_STM32F3'
# PLATFORM = 'CWHUSKY'    # CW313 + SAM4S 플랫폼 사용 시

### 공통 설정 파일 실행

`My_Setup.ipynb`를 실행하면 아래 작업이 자동으로 진행됩니다:

- ChipWhisperer 라이브러리 임포트 (`chipwhisperer`, `numpy`, `h5py`, `tqdm` 등)
- `scope` / `target` 객체 생성 및 USB 연결
- 타겟 펌웨어 컴파일 및 업로드
- 보조 함수 정의 (`my_fsr_cmd`, `my_get_trace`, `my_setting_num_samples` 등)

In [ ]:
# 공통 설정 노트북 실행 (scope, target 객체 초기화)
%run '../base/My_Setup.ipynb'

### 🎨 시각화 환경 (Bokeh) 초기화

본 강의의 모든 Trace 시각화는 **Bokeh**를 사용합니다.
matplotlib 정적 그래프와 달리 Bokeh는 다음과 같은 인터랙션을 제공해 Trace의 시점과 진폭을 확인하기 쉽습니다:

- 🔍 **Wheel Zoom** : 마우스 휠로 시간축을 자유롭게 확대/축소
- 🤚 **Pan** : 드래그로 영역 이동
- 📍 **Hover Tooltip** : 마우스 오버 시 정확한 (sample, value) 값 표시
- 🖼 **Box Zoom / Reset / Save** : 관심 구간만 박스로 선택해 확대, 원본 복귀, PNG 저장

In [ ]:
# Bokeh 시각화 초기화 (한 번만 실행하면 노트북 전체에서 인라인 출력 가능)
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
# BokehJS 를 CDN 이 아니라 출력에 직접 심는다. code-server 처럼 브라우저가
# cdn.bokeh.org 에 닿지 못하는 환경에서 그림이 빈 칸으로 보이는 것을 막는다.
from bokeh.resources import INLINE
from bokeh.models import HoverTool, ColumnDataSource, Range1d
from bokeh.palettes import Category10, Viridis256
from bokeh.layouts import column

output_notebook(INLINE)
print('✅ Bokeh 인라인 출력이 활성화되었습니다.')

---

# 📡 1단계 — 타겟 보드와 데이터 송수신

> **이 단계의 목표**
> ChipWhisperer가 SimpleSerial 프로토콜로 타겟 보드와 어떻게 대화하는지 익히고,
> 호스트가 보낸 데이터가 정확히 처리되어 돌아오는지 **Golden model(골든 모델)**과 비교해 검증합니다.

---

### 1.1 SimpleSerial 프로토콜 개요

ChipWhisperer는 **SimpleSerial 프로토콜**로 타겟과 통신합니다.
각 명령 패킷은 다음과 같은 고정 구조를 가집니다.

```
┌──────┬──────┬─────────┬──────────┬─────┐
│ cmd  │ scmd │  len    │  data[]  │ crc │
│(1B)  │(1B)  │  (1B)   │(최대245B)│(1B) │
└──────┴──────┴─────────┴──────────┴─────┘
```

| 필드 | 의미 |
|:----:|:----|
| `cmd`  | 명령 종류 — `0x81`(초기화/쓰기), `0x82`(연산실행), `0x83`(결과읽기) |
| `scmd` | 하위 명령 문자 — `'k'`(키), `'p'`(평문), `'l'`(길이), `'c'`(연산), `'r'`(결과) |
| `len`  | 데이터 바이트 수 |
| `data` | 실제 전송 데이터 |
| `crc`  | 오류 검출용 체크섬 |

### 1.2 보조 함수 `my_fsr_cmd()`

```python
my_fsr_cmd(target, cmd, scmd_char, data, payload_only=False)
```

- 위의 패킷 송신 + 응답 수신 + CRC 검증을 한 번에 처리하는 래퍼 함수입니다.
- `payload_only=True` → 수신 데이터 부분만 `bytearray`로 반환
- `payload_only=False`(기본) → 전체 응답 패킷 바이트 반환

이 함수의 정의는 `../base/My_Setup.ipynb`에 있으며, 본 단계에서는 **이해를 위해 raw `target.send_cmd()` / `target.read_cmd()`를 직접 호출**하는 방식으로 진행합니다.

### 1.3 테스트 데이터 준비

오늘 실험에서는 타겟 보드가 단순한 `data_k XOR data_p` 연산을 수행합니다.
임의의 245바이트 데이터를 생성해 통신 테스트를 합니다.

> 💡 **왜 245바이트인가?**
> SimpleSerial v2 패킷의 데이터 필드 최대 크기가 245바이트입니다.
> (cmd 1B + scmd 1B + len 1B + data 245B + crc 1B = 249B)
>
> > SimpleSerial V1 can send a maximum of 64 bytes of data per packet. SimpleSerial V2 can send a maximum of 249 bytes per packet.
> > — [chipwhisperer.readthedocs.io / simpleserial](https://chipwhisperer.readthedocs.io/en/latest/simpleserial.html)

In [ ]:
MAX_DATA_LEN = 245  # 한 번에 전송 가능한 최대 데이터 크기 (바이트)

# 재현성을 위해 시드 고정
random.seed(1)

# 무작위 키(key) 및 평문(plaintext) 데이터 생성
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))

print('=== 생성된 테스트 데이터 ===')
print(f'data_k (앞 16바이트): {data_k[:16].hex(" ")} ...')
print(f'data_p (앞 16바이트): {data_p[:16].hex(" ")} ...')
print(f'데이터 길이: {len(data_k)} 바이트')

### 1.4 데이터 전송: 키(k) 및 평문(p) 쓰기

타겟 보드의 메모리에 `data_k`와 `data_p`를 씁니다.
전송 후 타겟이 **에코(echo) 응답**을 보내오므로, 이를 통해 전송 성공 여부를 확인할 수 있습니다.

In [ ]:
# ── 키 데이터(data_k) 전송 ──────────────────────────────
# cmd=0x81(초기화/쓰기), scmd='k'(키), data=data_k

print('[ 키(k) 전송 ]')
target.flush()  # 수신 버퍼 초기화 (이전 잔여 데이터 제거)

target.send_cmd(cmd=0x81, scmd=ord('k'), data=data_k)
ret_k = target.read_cmd(timeout=2000)  # 최대 2초 대기

# 응답 패킷 파싱
# ret_k[0]       : cmd (0x81)
# ret_k[1]       : scmd (문자 'k'의 ASCII 값)
# ret_k[2]       : 데이터 길이
# ret_k[3:3+len] : 에코된 데이터
# ret_k[-2]      : CRC
print(f'  scmd 문자   : {chr(ret_k[1])}')
print(f'  데이터 길이 : {ret_k[2]} 바이트')
print(f'  에코 데이터 : {ret_k[3:3+ret_k[2]].hex(" ")} ...')
print(f'  CRC        : {hex(ret_k[-2])}')

In [ ]:
# ── 평문 데이터(data_p) 전송 ─────────────────────────────
# cmd=0x81(초기화/쓰기), scmd='p'(평문), data=data_p

print('[ 평문(p) 전송 ]')
target.flush()

target.send_cmd(cmd=0x81, scmd=ord('p'), data=data_p)
ret_p = target.read_cmd(timeout=2000)

print(f'  scmd 문자   : {chr(ret_p[1])}')
print(f'  데이터 길이 : {ret_p[2]} 바이트')
print(f'  에코 데이터 : {ret_p[3:3+ret_p[2]].hex(" ")} ...')
print(f'  CRC        : {hex(ret_p[-2])}')

### 1.5 출력 길이(l) 설정

타겟에게 **"결과를 몇 바이트 돌려줄 것인지"** 와 **"몇 바이트에 대해 연산할지"** 를 알려줍니다.

펌웨어의 `my_init()`은 `0x81 'l'` 페이로드 첫 바이트를 `global_len`에 저장합니다. 이후 `MY_OTP()`가 정확히 그 길이만큼 연산하고, `0x83 'r'` 응답도 같은 길이만큼 반환하므로 키·평문 길이와 일치시켜야 합니다.

In [ ]:
# ── 출력 길이 설정 ──────────────────────────────────────
# cmd=0x81, scmd='l', data=[MAX_DATA_LEN]  (1바이트 길이값)

print('[ 출력 길이(l) 설정 ]')
target.flush()

target.send_cmd(cmd=0x81, scmd=ord('l'), data=bytearray([MAX_DATA_LEN]))
ret_len = target.read_cmd(timeout=2000)

print(f'  scmd 문자             : {chr(ret_len[1])}')
print(f'  데이터 길이(항상 1)     : {ret_len[2]} 바이트')
print(f'  에코 데이터(설정된 길이) : {ret_len[3:3+ret_len[2]][0]} 바이트')
print(f'  CRC                  : {hex(ret_len[-2])}')

### 1.6 연산 실행 및 결과 수신

`cmd=0x82` 명령으로 타겟 연산을 트리거한 뒤,
`cmd=0x83` 명령으로 결과를 읽어옵니다.

> ⚠️ 연산 실행 명령(`0x82`)은 **응답을 기다리지 않습니다.**
> 트리거 신호만 발생시키고, 결과는 별도로 `0x83`으로 요청해야 합니다.
> (실제 부채널 측정에서는 이 사이에 `scope.capture()`가 들어갑니다 — 2단계에서 다룸)

In [ ]:
# ── 연산 실행 ───────────────────────────────────────────
# cmd=0x82('실행'), scmd='c'(compute), data=[] (없음)

print('[ 연산 실행 (k XOR p) ]')
target.flush()
target.send_cmd(cmd=0x82, scmd=ord('c'), data=[])
# 이 시점에 타겟 MCU가 연산을 수행합니다 (트리거 신호 발생)
print('  → 타겟 연산 완료')

In [ ]:
# ── 결과 읽기 ───────────────────────────────────────────
# cmd=0x83('결과읽기'), scmd='r'(result), data=[] (없음)

print('[ 결과 수신 ]')
target.flush()
target.send_cmd(cmd=0x83, scmd=ord('r'), data=[])
ret_k_XOR_p = target.read_cmd(timeout=2000)

print(f'  scmd 문자   : {chr(ret_k_XOR_p[1])}')
print(f'  데이터 길이 : {ret_k_XOR_p[2]} 바이트')
print(f'  결과 데이터 : {ret_k_XOR_p[3:3+ret_k_XOR_p[2]].hex(" ")} ...')
print(f'  CRC        : {hex(ret_k_XOR_p[-2])}')

### 1.7 Golden model(골든 모델)과 결과 비교

Python에서 직접 계산한 `k XOR p` 결과와 타겟의 결과를 비교해 통신 및 연산이 정확한지 검증합니다.

> 💡 **Golden model(골든 모델)**
> 타겟 하드웨어 없이 호스트 PC에서 동일한 연산을 수행한 기준 출력값입니다.
> 하드웨어 출력과 일치하면 **펌웨어와 통신 모두 정상**임을 의미합니다.

In [ ]:
# 타겟 보드에서 받은 결과
Return_k_XOR_p = ret_k_XOR_p[3 : 3 + ret_k_XOR_p[2]]

# 호스트(Python)에서 직접 계산한 골든 모델
Golden_k_XOR_p = bytes(x ^ y for x, y in zip(data_k, data_p))

print('=== 결과 비교 ===')
print(f'타겟 결과  : {Return_k_XOR_p.hex(" ")} ...')
print(f'골든 모델  : {Golden_k_XOR_p.hex(" ")} ...')
print()

if Golden_k_XOR_p == Return_k_XOR_p:
    print('✅ 통신 및 연산 검증 성공! (타겟 출력 == 골든 모델)')
else:
    print('❌ 불일치! 통신 오류 또는 펌웨어 오류를 확인하세요.')

---

# 📊 2단계 — Trace 수집 파라미터 설정

> **이 단계의 목표**
> 후속 분석의 신뢰도는 **Trace 수집 조건**에 좌우됩니다.
> 타겟 연산이 몇 클럭이나 걸리는지 측정해, 잘리지 않으면서도 메모리 낭비가 없는 적정 `samples` 값을 결정합니다.

---

### 2.1 개념 정리

```
                트리거 신호 (gpio4)
                    │
          ◄─────────┼───────────────────────────────►  시간 축
                    │
          │◄─pre──► │◄───── 연산 구간 (trig_count) ────►│
          │samples  │                                  │

  설정 공식 :  samples = (trig_count // decimate) + presamples
              여기서  trig_count ≈ adc_mul × (타겟 클럭 사이클 수)
```

| 파라미터 | 설명 |
|:--------:|:----|
| `samples`             | 실제로 저장할 전체 샘플 수 |
| `adc_src` & `adc_mul` | ADC 클럭 소스 및 샘플링 배수 |
| `presamples`          | 트리거 이전에 추가로 수집할 샘플 수 (트리거 직전 신호 관찰용) |
| `decimate`            | 다운샘플링 비율 (1=전체, 2=절반, …) |
| `trig_count`          | 트리거 ON → OFF 동안 ADC가 측정한 샘플 수 (읽기전용) |

**하드웨어별 최대 샘플 수**
- CW-Lite : 24,400
- CW1200  : 96,000
- CW-Husky: 131,070

### 2.2 ADC 클럭 설정

ChipWhisperer-Husky는 타겟 MCU 클럭에 **동기화된 ADC**를 사용합니다.

```python
scope.clock.adc_src  = 'clkgen_x4'   # MCU 클럭의 4배 속도로 샘플링
scope.clock.adc_mul  = 4             # CW-Pro/Husky 하드웨어별 설정 방법 차이

scope.adc.presamples = 0             # 프리샘플링 없음 (트리거 후 신호만 수집)
scope.adc.decimate   = 1             # 다운샘플링 없음 (전체 유지)

# ⚠ presamples와 decimate는 동시에 사용할 수 없습니다.
```

MCU가 7.37 MHz로 동작한다면 ADC는 **29.5 MHz**로 샘플링합니다.
클럭 1주기당 4개 Sample을 얻으므로 Trace의 시간 해상도가 높아지고, 분석 시 시점 식별이 쉬워집니다.

### 2.3 trig_count 측정 실험

스코프를 `arm()` 상태에서 한 번 연산을 실행하면, ChipWhisperer가 트리거 구간의 ADC 샘플 수를 자동으로 측정합니다.
이 값을 보고 `samples`를 계산합니다.

In [ ]:
# ── 스코프 기본 설정 및 타겟 보드 리셋 ──────────────────

scope.default_setup()  # 스코프 초기화 (트리거, 게인, 샘플 수 등 기본값 로드)

scope.clock.adc_src = 'clkgen_x4'   # 'clkgen_x4' 또는 'clkgen_x1'
scope.clock.adc_mul = 4

RATIO_DECIMATE = 1    # 다운샘플링 비율
NUM_PRESAMPLES = 0    # 트리거 이전 프리샘플 수 (decimate=1일 때만 사용 가능)

# reset_target(scope): 타겟 MCU를 하드 리셋하는 함수
# → ../base/Setup_Generic.ipynb 에 정의
reset_target(scope)
time.sleep(1)  # 리셋 후 부팅 대기
reset_target(scope)
time.sleep(1)

print('스코프 및 타겟 초기화 완료')

In [ ]:
# ── 테스트용 데이터 준비 ────────────────────────────────
MAX_DATA_LEN = 245  # 전송 데이터 크기

random.seed(1)
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))

print(f'MAX_DATA_LEN   : {MAX_DATA_LEN}')
print(f'RATIO_DECIMATE : {RATIO_DECIMATE}')
print(f'NUM_PRESAMPLES : {NUM_PRESAMPLES}')

In [ ]:
# ── 타겟에 데이터 주입 ──────────────────────────────────
my_fsr_cmd(target, 0x81, 'k', data_k)
my_fsr_cmd(target, 0x81, 'p', data_p)
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))

# ── 스코프 무장(arm) → 연산 실행 → 캡처 대기 ────────────
scope.arm()                          # 트리거 대기 상태로 진입
my_fsr_cmd(target, 0x82, 'c', [])    # 연산 실행 (트리거 발생)

if scope.capture():                  # Trace 캡처 (타임아웃 시 True 반환)
    print('❌ 타겟 타임아웃!')
    sys.exit('Target timed out!')

# trig_count: 트리거 ON → OFF 동안의 ADC 샘플 수
print(f'\n⏱ 연산 구간 ADC 샘플 수 (trig_count) : {scope.adc.trig_count} samples')

# ── 샘플 수 계산 ────────────────────────────────────────
scope.adc.decimate = RATIO_DECIMATE
if RATIO_DECIMATE == 1:
    scope.adc.presamples = NUM_PRESAMPLES

scope.adc.samples = (scope.adc.trig_count // scope.adc.decimate) + scope.adc.presamples

print(f'   presamples              : {scope.adc.presamples}')
print(f'   decimate                : {scope.adc.decimate}')
print(f'✅ 최종 수집 샘플 수 (samples) : {scope.adc.samples}')

> 💡 **`samples` 설정 공식 해설**
>
> $$
> \text{samples} = \left\lfloor \frac{\text{trig\_count}}{\text{decimate}} \right\rfloor + \text{presamples}
> $$
>
> - `trig_count`  : 실제 연산에 걸린 ADC 샘플 수 (자동 측정)
> - `decimate`    : 다운샘플링 비율 (1이면 전체, 2이면 절반)
> - `presamples`  : 트리거 전에 추가할 여유 샘플 수
>
> **너무 짧게 설정** → Trace 끝이 잘려 분석에 사용할 수 없습니다.
> **너무 길게 설정** → 메모리 낭비 및 분석 속도 저하.

---

# 🌊 3단계 — 단일 Trace 수집 및 시각화

> **이 단계의 목표**
> Trace 1개를 수집하고, **Bokeh 인터랙티브 그래프**로 시각화해 잘림·클리핑·트리거 정렬 상태를 점검합니다.

---

### 3.1 `my_setting_num_samples()` 함수

2단계의 `trig_count` 측정 + `samples` 설정을 한 번에 처리하는 래퍼 함수입니다.

```python
MAX_TR_LEN = my_setting_num_samples(target, scope,
                                    NUM_PRESAMPLES=NUM_PRESAMPLES,
                                    RATIO_DECIMATE=RATIO_DECIMATE)
# → 내부적으로 더미 연산을 실행해 trig_count 측정
# → scope.adc.samples를 자동 설정
# → 최종 samples 값을 반환 (DB 배열 크기 사전 결정에 사용)
# → NUM_PRESAMPLES 또는 RATIO_DECIMATE 둘 중 하나만 설정 가능 (동시 사용 X)
```

In [ ]:
# ── 스코프 수집 파라미터 재설정 ─────────────────────────
scope.default_setup()

RATIO_DECIMATE = 1
NUM_PRESAMPLES = 0
scope.clock.adc_src = 'clkgen_x4'
scope.clock.adc_mul = 4

reset_target(scope)
reset_target(scope)

# ── 테스트 데이터 준비 ──────────────────────────────────
MAX_DATA_LEN = 245
random.seed(1)
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))

# 타겟에 데이터 주입
my_fsr_cmd(target, 0x81, 'k', data_k)
my_fsr_cmd(target, 0x81, 'p', data_p)
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))

# samples 자동 설정 및 반환
MAX_TR_LEN = my_setting_num_samples(target, scope,
                                    NUM_PRESAMPLES=NUM_PRESAMPLES,
                                    RATIO_DECIMATE=RATIO_DECIMATE)
print(f'\n파형 1개당 샘플 수 (MAX_TR_LEN): {MAX_TR_LEN}')

### 3.2 Trace 1개 수집

`my_get_trace(target, scope)` 함수는 내부적으로 다음을 순서대로 실행합니다:

1. `scope.arm()` — 트리거 대기
2. `my_fsr_cmd(... 0x82 ...)` — 연산 실행 (트리거 발생)
3. `scope.capture()` — Trace 수신
4. `my_fsr_cmd(... 0x83 ...)` — 결과 읽기

반환값은 `(타겟 출력값, Trace를 담은 NumPy 배열)` 튜플입니다.

In [ ]:
# ── Trace 1개 수집 ───────────────────────────────────────
try:
    data_o, trace = my_get_trace(target, scope)
    # data_o : 타겟 연산 결과 (bytearray)
    # trace  : 전력 Trace (NumPy 배열, 길이 = MAX_TR_LEN)
except (TimeoutError, ValueError) as e:
    print(f'오류 발생: {e}')

print(f'출력값      : {data_o.hex(" ")} ...')
print(f'파형 길이    : {len(trace)} 샘플')
print(f'파형 최솟값  : {trace.min():.4f}')
print(f'파형 최댓값  : {trace.max():.4f}')

### 3.3 Bokeh로 Trace 시각화

수집된 Trace를 인터랙티브 그래프로 그립니다.
**그래프 위에서 마우스를 움직이면 정확한 (sample, 전력값)이 표시**되고, 휠로 자유롭게 확대/축소할 수 있습니다.

In [ ]:
# ── Bokeh 인터랙티브 Trace 시각화 ────────────────────────
x = np.arange(len(trace))

source = ColumnDataSource(data=dict(x=x, y=trace))

p = figure(
    width=900, height=320,
    title='Power Trace — Single Acquisition',
    x_axis_label='Sample Index',
    y_axis_label='Amplitude (V)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
    border_fill_color='white',
)

# 본 Trace
p.line('x', 'y', source=source,
       line_width=1.2, line_color='#2E86AB', line_alpha=0.9)

# 시각적 다듬기
p.title.text_font_size = '13pt'
p.title.text_color = '#2c3e50'
p.title.align = 'center'
p.grid.grid_line_alpha = 0.3
p.xaxis.axis_label_text_font_style = 'normal'
p.yaxis.axis_label_text_font_style = 'normal'
p.outline_line_color = None

# Hover 툴팁 추가
hover = HoverTool(
    tooltips=[
        ('Sample', '@x{0}'),
        ('Amplitude', '@y{0.0000}'),
    ],
    mode='vline',
)
p.add_tools(hover)

show(p)

> 🔬 **Trace를 보면서 점검할 사항**
>
> - Trace가 중간에 잘리지 않는가? → `samples` 값 재조정 필요
> - 트리거 직전 flat 구간이 보이는가? → `presamples` 적용 효과 확인
> - 진폭(amplitude)이 너무 크거나 작지 않은가? → `scope.gain` 조정 필요
> - 명확한 클럭 패턴(주기적 봉우리)이 보이는가? → 동기 ADC가 정상 작동 중

---

# 💾 4단계 — 다량의 Trace를 HDF5 파일로 저장

> **이 단계의 목표**
> 이 실습의 후속 통계 분석에는 여러 Trace가 필요하며, 필요한 수는 시험 목적과 효과 크기에 따라 달라집니다.
> 메모리에 한꺼번에 올리지 않고, Dataset의 각 Attributes를 담는 **HDF5 배열에 행을 점진적으로 추가(append)**해 저장합니다.

---

### 4.1 HDF5 포맷 개요

**HDF5(Hierarchical Data Format 5)** 는 대용량 과학 데이터 저장에 특화된 파일 형식입니다.
파이썬의 `h5py` 라이브러리로 접근하며, 여러 Trace와 관련 Attributes를 한 파일에서 관리할 수 있습니다.

```
SCA_DB.h5
├── i_k  [N × data_len]   uint8     ← 입력 키 배열
├── i_p  [N × data_len]   uint8     ← 입력 평문 배열
├── o    [N × data_len]   uint8     ← 연산 출력 배열
└── t    [N × tr_len]     float32   ← 전력 Trace 배열
```

`maxshape=(None, ...)`으로 설정해 **N(Record 수)을 동적으로 확장**할 수 있게 합니다.
`chunks=True`는 청크 단위 저장을 활성화해 random access I/O 성능을 높입니다.

### 4.2 TV(Test Vector) 설정

| `TV_case`  | 설명 | 용도 |
|:---------:|:----:|:----|
| `'fixed'`  | 매번 동일한 k, p 사용 | 동일 입력에서 Trace 반복성과 수집 안정성 확인 |
| `'random'` | 매번 무작위 k, p 사용 | 다양한 입력을 탐색하는 Dataset 수집. 키가 Record마다 바뀌므로 일반적인 고정 키 CPA용 Dataset은 아님 |

In [ ]:
# ══════════════════════════════════════════════════════════
#  스코프 초기화 및 파라미터 설정
# ══════════════════════════════════════════════════════════
scope.default_setup()

RATIO_DECIMATE = 1
NUM_PRESAMPLES = 0
scope.clock.adc_src = 'clkgen_x4'
scope.clock.adc_mul = 4

reset_target(scope)
reset_target(scope)

print('스코프 및 타겟 초기화 완료')

In [ ]:
# ══════════════════════════════════════════════════════════
#  데이터 길이 및 samples 자동 결정
# ══════════════════════════════════════════════════════════
MAX_DATA_LEN = 16   # 실제 암호 연산 데이터 크기 (e.g., AES-128이면 16바이트)

# 타겟 초기화 (더미 데이터로 레지스터 설정)
my_fsr_cmd(target, 0x81, 'k', bytearray(MAX_DATA_LEN))   # 16바이트 영벡터
my_fsr_cmd(target, 0x81, 'p', bytearray(MAX_DATA_LEN))
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))

# 실제 연산 길이를 측정해 scope.adc.samples를 설정하고 그 값을 반환
MAX_TR_LEN = my_setting_num_samples(target, scope,
                                    NUM_PRESAMPLES=NUM_PRESAMPLES,
                                    RATIO_DECIMATE=RATIO_DECIMATE)

print(f'\n=== 파형 수집 파라미터 ===')
print(f'  데이터 크기   : {MAX_DATA_LEN} 바이트')
print(f'  프리샘플 수   : {NUM_PRESAMPLES}')
print(f'  파형 샘플 수  : {MAX_TR_LEN}')

### 4.3 Trace 수집 메인 루프

아래 셀을 실행하면 다음이 순차적으로 진행됩니다:

1. HDF5 파일 (`tmp_SCA_DB.h5`) 생성 및 빈 배열 초기화
2. `NUM_OF_TRACES`만큼 반복:
   - TV 생성 (`fixed` 또는 `random`)
   - 타겟에 키/평문 주입 + 에코 검증
   - `my_get_trace()`로 Trace 수집
   - 동일한 Record를 이루는 각 HDF5 배열에 행 1개씩 append
3. 정상 완료 시 타임스탬프가 붙은 최종 파일명으로 rename

> 💡 **`debug_mode = True`**로 두면 Trace 5개만 수집해 메모리에 보관하고 화면에 표시하며, HDF5 파일에는 저장하지 않습니다.
> 본격 수집 전 동작 점검용으로 활용하세요.

In [ ]:
# ══════════════════════════════════════════════════════════
#  Trace 수집 메인 루프
# ══════════════════════════════════════════════════════════

debug_mode    = False        # True: 소량 수집 + 화면 출력 / False: 실제 DB 저장
NUM_OF_TRACES = 10           # 수집 목표 Trace 수
FileDir       = '../traces'  # 저장 경로
TV_case       = 'random'     # 'fixed' 또는 'random'

# ── 디버그 모드: 소량 수집 후 Trace만 시각화 ─────────────
if debug_mode:
    NUM_OF_TRACES = 5
    array_i_k, array_i_p, array_o, array_t = [], [], [], []

# ── 이 파일이 따르는 규약 ────────────────────────────────
# 저장 구조는 저장소 루트의 SCHEMA.md 를 따른다. 용어는 GLOSSARY.md 가 정본이다.
# 핵심만 적으면:
#   - 파일 하나 = Dataset 하나 (Target 1개 × Channel 1개)
#   - /<subset>/ 아래에 같은 수집 규약으로 받은 Record 를 모은다
#   - 배열 이름은 trace / key / plaintext / ciphertext 로 고정
#   - 측정 조건은 루트 HDF5 attrs 에 적는다 — 나중에 이 파일만 받은 사람도
#     "무엇을 어떻게 쟀는지" 알 수 있어야 하기 때문이다
#
# subset 이름은 자유지만 role 은 SCHEMA.md §4.1 목록에서 고른다.
# 여기서는 키·평문을 둘 다 무작위로 돌리면 exploration, 둘 다 고정이면
# 고정 입력 집단(leakage-detection-fixed)에 해당한다.
SUBSET = 'explore' if TV_case == 'random' else 'fixed_input'
ROLE   = 'exploration' if TV_case == 'random' else 'leakage-detection-fixed'

# ── HDF5 파일 생성 (동적 확장 가능한 배열 준비) ──────────
with h5py.File(f'{FileDir}/tmp_SCA_DB.h5', 'w') as f:

    # ── 루트 Metadata (SCHEMA.md §3) ────────────────────
    # 모르는 값은 적지 않는다. 추정치를 넣으면 다음 사람이 그것을 실측값으로 오해한다.
    f.attrs['schema']                 = 'sca-hdf5'
    f.attrs['schema_version']         = '1.0'

    f.attrs['target_name']            = PLATFORM
    f.attrs['target_device']          = 'STM32F303'
    f.attrs['target_clock_hz']        = float(scope.clock.clkgen_freq)
    f.attrs['iut_algorithm']          = 'OTP (bytewise XOR)'
    f.attrs['iut_implementation']     = 'simpleserial_main/my_crypt.c MY_OTP'
    f.attrs['iut_countermeasure']     = 'none'

    f.attrs['channel_type']           = 'power'
    f.attrs['channel_probe']          = 'CW308 SHUNTL (내장 션트)'
    f.attrs['channel_gain_db']        = float(scope.gain.db)

    f.attrs['sample_rate_hz']         = float(scope.clock.adc_freq)
    # CW-Lite의 10-bit ADC를 전제로 한다. 다른 스코프를 쓰면 수집 전에 실제
    # ADC 해상도에 맞게 이 Metadata 값을 바꿔야 Dataset이 측정 조건을 정확히 나타낸다.
    f.attrs['sample_resolution_bits'] = 10
    f.attrs['samples_per_trace']      = int(MAX_TR_LEN)
    f.attrs['sample_dtype']           = 'float32'
    # 이미 정규화된 실수라 나눌 필요가 없다. 값의 물리 단위(V)는 션트·이득을
    # 모두 알아야 정해지므로 여기서는 단정하지 않는다 (SCHEMA.md §5.2).
    f.attrs['sample_scale']           = 1.0

    f.attrs['trigger_source']         = 'tio4 (GPIO4)'
    f.attrs['trigger_semantics']      = 'MY_OTP 연산 구간 (0x82 c)'
    f.attrs['trigger_samples']        = int(scope.adc.trig_count)

    f.attrs['alignment']              = 'none'      # 트리거 동기만, 후처리 정렬 없음

    f.attrs['acquisition_start']      = time.strftime('%Y-%m-%dT%H:%M:%S')
    f.attrs['tool_chain']             = f'chipwhisperer {cw.__version__}'

    # ── Subset 과 Attributes (SCHEMA.md §2) ─────────────
    g = f.create_group(SUBSET)
    g.attrs['role']     = ROLE
    g.attrs['key_mode'] = TV_case
    g.attrs['pt_mode']  = TV_case

    # shape=(0, ...)        : 초기 행 수 0 (아직 비어있음)
    # maxshape=(None, ...)  : 행 방향으로 무제한 확장 허용
    # chunks=True           : 청크 단위 저장으로 I/O 성능 향상
    dset_key = g.create_dataset('key', shape=(0, MAX_DATA_LEN),
                                maxshape=(None, MAX_DATA_LEN), dtype='uint8', chunks=True)
    dset_plaintext = g.create_dataset('plaintext', shape=(0, MAX_DATA_LEN),
                                maxshape=(None, MAX_DATA_LEN), dtype='uint8', chunks=True)
    dset_ciphertext   = g.create_dataset('ciphertext', shape=(0, MAX_DATA_LEN),
                                maxshape=(None, MAX_DATA_LEN), dtype='uint8', chunks=True)
    dset_trace   = g.create_dataset('trace', shape=(0, MAX_TR_LEN),
                                maxshape=(None, MAX_TR_LEN),   dtype='float32', chunks=True)

    # ── 수집 루프 ────────────────────────────────────────
    for i in trange(NUM_OF_TRACES, desc='파형 수집 중'):

        # TV 생성
        if TV_case == 'fixed':
            data_i_k = bytearray(b'\xA0' * MAX_DATA_LEN)  # 고정 키
            data_i_p = bytearray(b'\x08' * MAX_DATA_LEN)  # 고정 평문
        elif TV_case == 'random':
            data_i_k = bytearray([random.randint(0, 255) for _ in range(MAX_DATA_LEN)])
            data_i_p = bytearray([random.randint(0, 255) for _ in range(MAX_DATA_LEN)])
        else:
            print('TV_case 오류: fixed 또는 random만 허용됩니다')
            break

        # 타겟 초기화 및 데이터 주입 검증
        # my_fsr_cmd(..., payload_only=True) → 에코 데이터만 bytearray로 반환
        if (
            data_i_k != my_fsr_cmd(target, 0x81, 'k', data_i_k, payload_only=True) or
            data_i_p != my_fsr_cmd(target, 0x81, 'p', data_i_p, payload_only=True) or
            MAX_DATA_LEN != my_fsr_cmd(target, 0x81, 'l',
                                       bytearray([MAX_DATA_LEN]), payload_only=True)[0]
        ):
            print(f'[{i}] 초기화 실패! 해당 파형 건너뜀')
            continue  # 실패한 Record를 저장하지 않고 다음 반복으로 진행

        # Trace 수집
        try:
            data_o, trace = my_get_trace(target, scope)
        except (TimeoutError, ValueError) as e:
            print(f'[{i}] 수집 오류: {e}')
            continue

        # ── 디버그 모드: 메모리에만 저장 ─────────────────
        if debug_mode:
            print(f'  [{i}] i_k={data_i_k.hex(" ")}')
            print(f'       i_p={data_i_p.hex(" ")}')
            print(f'       o  ={data_o.hex(" ")}\n')
            array_i_k.append(data_i_k)
            array_i_p.append(data_i_p)
            array_o.append(data_o)
            array_t.append(trace)

        # ── 실제 모드: HDF5에 즉시 저장 (메모리 절약) ────
        else:
            # resize → 행을 1개 늘림
            dset_key.resize(dset_key.shape[0] + 1, axis=0)
            dset_plaintext.resize(dset_plaintext.shape[0] + 1, axis=0)
            dset_ciphertext.resize(dset_ciphertext.shape[0]   + 1, axis=0)
            dset_trace.resize(dset_trace.shape[0]   + 1, axis=0)

            # 마지막 행에 데이터 기록
            dset_key[-1:] = np.array(data_i_k)
            dset_plaintext[-1:] = np.array(data_i_p)
            dset_ciphertext[-1:]   = np.array(data_o)
            dset_trace[-1:]   = np.array(trace)

    # 실제 Record 수를 기록한다. 수집 실패로 건너뛴 시행이 있을 수 있어
    # NUM_OF_TRACES 와 다를 수 있으므로, 목표치가 아니라 실측값을 적는다.
    if not debug_mode:
        g.attrs['n_records'] = int(dset_trace.shape[0])

# ── 수집 완료 후 처리 ────────────────────────────────────
if debug_mode:
    # 수집한 Trace 전체를 Bokeh로 겹쳐 시각화
    arr_t = np.array(array_t)
    p = figure(
        width=900, height=320,
        title=f'{len(array_t)} Power Traces (debug_mode, TV_case={TV_case})',
        x_axis_label='Sample Index',
        y_axis_label='Amplitude (정규화)',
        tools='pan,wheel_zoom,box_zoom,reset,save',
        active_scroll='wheel_zoom',
        background_fill_color='#fafafa',
    )
    palette = Category10[10]
    x = np.arange(arr_t.shape[1])
    for i in range(len(array_t)):
        p.line(x, arr_t[i], line_width=1.0,
               line_color=palette[i % 10], line_alpha=0.7,
               legend_label=f'Trace {i}')
    p.legend.location = 'top_right'
    p.legend.click_policy = 'hide'
    p.title.text_font_size = '13pt'
    p.grid.grid_line_alpha = 0.3
    p.outline_line_color = None
    show(p)
else:
    # 타임스탬프로 파일명 생성 → 파일 이름 변경
    d    = time.strftime('%Y%m%d_%H%M%S', time.localtime(time.time()))
    file = Path(f'{FileDir}/tmp_SCA_DB.h5')
    file.rename(f'{FileDir}/{d}_SCA_DB.h5')
    print(f'\n✅ 저장 완료: {FileDir}/{d}_SCA_DB.h5')

print('\nDone!')

In [ ]:
# 측정 완료 후 스코프 및 타겟 연결 해제
scope.dis()
target.dis()
print('연결 해제 완료')

---

# 📂 5단계 — HDF5 파일에서 Dataset 불러오기 및 확인

> **이 단계의 목표**
> 4단계에서 저장한 `.h5` 파일을 다시 불러와, 데이터 구조와 Trace 배열의 형태를 확인합니다. 값의 정확성이나 스키마 준수 여부는 별도의 검증기를 실행해야 합니다.
> ChipWhisperer 하드웨어가 없는 환경에서도 이 단계부터는 동일하게 실행할 수 있습니다.

---

### 5.1 파일 불러오기

In [ ]:
import h5py
import numpy as np
from pathlib import Path

# ── 파일 경로 설정 ──────────────────────────────────────
FileDir   = '../traces'
file_path = Path(FileDir) / '20260427_143337_SCA_DB.h5'  # ← 실제 파일명으로 수정

# ── 데이터 로드 ─────────────────────────────────────────
# 구조는 SCHEMA.md 를 따른다: 루트 attrs 에 측정 조건, /<subset>/ 아래에 배열.
with h5py.File(file_path, 'r') as f:

    print('=== Metadata (측정 조건) ===')
    for k in sorted(f.attrs):
        print(f'  {k:<24} {f.attrs[k]}')

    # subset 은 이름이 자유이므로 목록에서 첫 번째를 고른다.
    # 여러 개면 role 을 보고 목적에 맞는 것을 고르면 된다.
    subsets = list(f)
    print(f'\n=== Subset : {subsets} ===')
    g = f[subsets[0]]
    print(f'  role     : {g.attrs.get("role", "(없음)")}')
    print(f'  key_mode : {g.attrs.get("key_mode", "(없음)")}'
          f'   pt_mode : {g.attrs.get("pt_mode", "(없음)")}')

    i_k = g['key'][:]         # shape: (N, data_len)
    i_p = g['plaintext'][:]   # shape: (N, data_len)
    o   = g['ciphertext'][:]  # shape: (N, data_len)
    t   = g['trace'][:]       # shape: (N, tr_len)

print('\n=== 배열 크기 ===')
print(f'  key        : {i_k.shape}  (레코드 수 × 데이터 길이)')
print(f'  plaintext  : {i_p.shape}')
print(f'  ciphertext : {o.shape}')
print(f'  trace      : {t.shape}   (레코드 수 × 샘플 수)')

# 행 정렬 규칙: 모든 배열의 i 번째 행이 같은 Execution 에 대응한다.
assert i_k.shape[0] == i_p.shape[0] == o.shape[0] == t.shape[0], '행 수 불일치'
print('\n행 정렬 OK — i 번째 행이 모두 같은 실행(Execution)이다.')

### 5.2 개별 샘플 검증

저장된 데이터가 `o = i_k XOR i_p` 관계를 정확히 만족하는지 확인합니다.

In [ ]:
idx = 0  # 확인할 샘플 인덱스

print(f'=== 샘플 #{idx} ===')
print(f'  i_k : {bytes(i_k[idx]).hex(" ")}')
print(f'  i_p : {bytes(i_p[idx]).hex(" ")}')
print(f'  o   : {bytes(o[idx]).hex(" ")}')

# 결과 검증: o == i_k XOR i_p 인지 확인
expected = bytes(x ^ y for x, y in zip(i_k[idx], i_p[idx]))
match = bytes(o[idx]) == expected
print(f'\n  검증(o == k XOR p): {"✅ 일치" if match else "❌ 불일치"}')

### 5.3 Bokeh로 여러 Trace 시각화

수집된 여러 Trace를 겹쳐서 시각화합니다.
범례를 클릭하면 해당 Trace를 숨기거나 다시 표시할 수 있습니다.

In [ ]:
# ── 여러 Trace의 Bokeh 시각화 ────────────────────────────
NUM_PLOT = 5  # 겹쳐 그릴 Trace 수

x = np.arange(t.shape[1])

p = figure(
    width=900, height=360,
    title=f'Power Traces Overlay (first {NUM_PLOT} traces)',
    x_axis_label='Sample Index',
    y_axis_label='Amplitude (V)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
    border_fill_color='white',
)

palette = Category10[10]
for i in range(NUM_PLOT):
    p.line(x, t[i],
           line_width=1.0,
           line_color=palette[i % 10],
           line_alpha=0.75,
           legend_label=f'Trace {i}')

# 시각적 다듬기
p.title.text_font_size = '13pt'
p.title.text_color = '#2c3e50'
p.title.align = 'center'
p.grid.grid_line_alpha = 0.3
p.xaxis.axis_label_text_font_style = 'normal'
p.yaxis.axis_label_text_font_style = 'normal'
p.outline_line_color = None

# 범례 설정
p.legend.location = 'top_right'
p.legend.click_policy = 'hide'  # 범례 클릭 시 해당 Trace 숨김
p.legend.label_text_font_size = '9pt'
p.legend.background_fill_alpha = 0.7

# Hover 툴팁
hover = HoverTool(tooltips=[('Sample', '$x{0}'), ('Amplitude', '$y{0.0000}')])
p.add_tools(hover)

show(p)
print('\n완료!')

> 💡 **Trace 겹쳐 보기(Overlay)의 의미**
>
> - 여러 Trace의 특징점이 **잘 정렬됨** → 트리거와 수집 타이밍의 반복성을 확인하는 단서
> - Trace가 **어긋나거나 흔들림** → 트리거 지터, 클럭 또는 샘플레이트 설정 점검 필요
> - **Fixed TV에서 Trace가 거의 동일** → 반복 수집의 정렬 상태를 점검할 단서. CPA/DPA 가능 여부는 평문 다양성, Trace 수, SNR 등을 별도로 확인해야 함
>
> Bokeh의 wheel zoom으로 특정 시점을 확대하면 같은 Sample 위치에서 Trace 간 진폭 차이를 확인할 수 있습니다.
> 이 차이에는 데이터 의존 누설뿐 아니라 측정 Noise도 섞일 수 있습니다. 누설 여부는 CPA, DPA 또는 t-test 같은 통계 분석으로 구분해야 합니다.

---

## 📝 1강 요약

| 단계 | 핵심 함수 / 명령 | 결과 |
|:----:|:---|:---|
| 1 | `target.send_cmd()` / `read_cmd()` / `my_fsr_cmd()` | 데이터 송수신 및 골든 모델 검증 |
| 2 | `scope.arm()` → `scope.capture()` → `trig_count` | 연산 구간 측정 및 `samples` 결정 |
| 3 | `my_get_trace()` + Bokeh `figure().line()` | 단일 Trace 수집 및 인터랙티브 시각화 |
| 4 | HDF5 배열 + `resize` + `trange` | 여러 Record를 동적으로 파일에 저장 |
| 5 | `h5py.File('r')` + Bokeh overlay | HDF5 파일 불러오기 및 여러 Trace 비교 |

### ✅ 1강에서 익혀야 할 핵심 개념

1. **SimpleSerial 패킷 구조** (cmd / scmd / len / data / crc)
2. **trig_count → samples 공식** : `samples = (trig_count // decimate) + presamples`
3. 크기 확장이 가능한 **HDF5 배열**로 메모리 효율적 저장
4. **TV(Test Vector)** 의 의미와 `fixed` / `random`의 용도 차이
5. **Trace 시각화**를 통한 수집 상태 점검 (잘림, 진폭, 트리거 타이밍)

---
*ChipWhisperer 입문 강의 — 1강 끝*